# Medical NER — bản NHẸ + AN TOÀN. Kaggle GPU.

**TRƯỚC `Run All`:** Settings → **GPU T4** (1 con đủ) + **Internet ON**.
**Thời gian ~1h** (không còn 9h). Tải **`output_ner.zip`** để nộp.

Điểm mấu chốt an toàn:
- **base** (`xlm-roberta-base`) nhanh 4-5× large; DAPT vẫn là đòn khép domain gap.
- **Ép 1 GPU** — 2 GPU làm HF Trainer CHẬM HƠN (DataParallel overhead).
- **Nộp baseline NGAY sau test** → luôn có file nộp dù train hỏng.


In [ ]:
# 1) Clone repo + ÉP 1 GPU (tránh DataParallel làm chậm)
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO = "https://github.com/Khanhhh239/Medical_Retrieve.git"
D = "/kaggle/working/Medical_Retrieve"
if not os.path.isdir(D):
    subprocess.run(["git","clone","-q",REPO,D], check=True)
else:
    subprocess.run(["git","-C",D,"pull","-q"])
os.chdir(D); print('cwd:', os.getcwd())


In [ ]:
# 2) Deps
!pip install -q rapidfuzz pyyaml regex accelerate
import rapidfuzz, yaml, regex, torch, transformers
print("torch",torch.__version__,"| CUDA",torch.cuda.is_available(),"| GPUs",torch.cuda.device_count())
print("transformers",transformers.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# 3) Test nhanh
!python -m pytest -q 2>&1 | tail -3


In [ ]:
# 4) *** LƯỚI AN TOÀN *** — nộp BASELINE ngay (không cần model). ~30s
#    Nếu mọi thứ sau đây hỏng/timeout, VẪN có output.zip để nộp.
!python scripts/build_kb.py --icd || echo 'icd skip'
!python scripts/predict.py --pipeline baseline --out_dir output --zip
import shutil, os
if os.path.exists("output.zip"): shutil.copy("output.zip","/kaggle/working/output.zip"); print("SAFETY: output.zip san sang")


In [ ]:
# 5) DAPT — MLM trên 100 file test THẬT (base, 3 epochs). ~5'
!python scripts/dapt.py --model xlm-roberta-base --epochs 3 --batch_size 16 --out models/dapt


In [ ]:
# 6) Sinh data template (nhẹ)
!python scripts/gen_synth.py --n_train 4000 --n_dev 400


In [ ]:
# 7) RxNav (thuốc) — dùng cache repo, chỉ query mã mới
!python scripts/build_kb.py --rxnorm || echo 'rxnorm skip -> dung cache'


In [ ]:
# 8) FINE-TUNE NER — khởi từ DAPT, base, batch 16 (1 T4 vừa). ~30-40'
!python scripts/train_ner.py --model models/dapt \
   --train data/synthetic/train.jsonl \
   --epochs 3 --batch_size 16 --grad_accum 1 --max_length 256 --optim adamw_torch \
   --out models/ner


In [ ]:
# 9) Inference NER -> output_ner.zip (BẢN NỘP CHÍNH)
!python scripts/predict.py --pipeline ner --model_dir models/ner --out_dir output_ner --zip
import shutil, os
if os.path.exists("output_ner.zip"):
    shutil.copy("output_ner.zip","/kaggle/working/output_ner.zip")
    print(">>> TAI output_ner.zip o panel Output de NOP <<<")
else:
    print("!!! NER hong -> nop output.zip (baseline) o cell 4")


In [ ]:
# 10) So file văn xuôi 8.txt: baseline vs NER (tham khảo)
import json
try:
    print("BASELINE 8:", json.dumps(json.load(open("output/8.json")),ensure_ascii=False)[:250])
    print("NER      8:", json.dumps(json.load(open("output_ner/8.json")),ensure_ascii=False)[:250])
except Exception as e: print(e)


## Ghi chú
- **base thay large**: score chênh nhỏ, nhưng **chạy XONG** > large timeout mất trắng. DAPT mới là đòn chính.
- **1 GPU cố ý**: 2×T4 + HF Trainer = DataParallel, CHẬM hơn với model này.
- **Baseline nộp trước** (cell 4) = lưới an toàn, luôn có file.
- Muốn mạnh hơn khi đã ổn định + còn nhiều quota: `--model xlm-roberta-large`, `--n_train 8000`, DAPT epochs 6.
- OOM (khó xảy ra với base): giảm `--batch_size 8`.
